# Stage-Wise Curvature Tuning on Flowers102

## Environment Setup

This notebook is designed to run independently in Google Colab.

The repository is cloned or updated, the `stage-wise-ct` branch is selected, and the package versions used for the research experiments are installed.

After dependency installation, the Colab runtime must be restarted before continuing with the experiment cells.

In [ ]:
!nvidia-smi

%cd /content

import os

if not os.path.exists("/content/curvature-tuning-research"):
    !git clone -b stage-wise-ct https://github.com/Syed1611/curvature-tuning-research.git
else:
    print("Repository already exists; skipping clone.")

%cd /content/curvature-tuning-research
!git checkout stage-wise-ct
!git pull

%cd /content/curvature-tuning-research/src/curvature-tuning
!git branch --show-current

In [ ]:
%pip install -q -r requirements.txt
%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

### Runtime Restart Required

After installing the dependencies and NumPy 1.26.3, restart the Colab runtime so that the required package versions are loaded correctly.

**After restarting, do not rerun the installation cells above. Continue from the next cell.**

In [ ]:
%cd /content/curvature-tuning-research/src/curvature-tuning

import os
os.environ["WANDB_MODE"] = "disabled"

print("Working directory:", os.getcwd())

In [ ]:
import torch
import datasets
import numpy
import pandas
import sklearn
import tqdm
import loguru

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("datasets:", datasets.__version__)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("sklearn:", sklearn.__version__)
print("tqdm:", tqdm.__version__)
print("loguru:", loguru.__version__)

# Cross-Dataset Evaluation: Flowers102

Flowers102 is used as the third downstream classification dataset for evaluating Stage-Wise Curvature Tuning.

The original frozen Baseline and S-CT methods are first evaluated using seeds 42, 43, and 44. S-CT retains the original validation-based search over β values from 0.70 to 1.00.

The proposed 4-parameter SW-CT method is then evaluated using the configuration previously selected during the Beans experiments:

- Initial β = 0.78
- β learning rate = 0.01
- Four trainable stage-wise β parameters
- Fixed CTU coefficient c = 0.5

No Flowers-specific SW-CT hyperparameter tuning is performed. This allows Flowers102 to serve as an additional cross-dataset evaluation of the existing SW-CT configuration.

In [ ]:
%cd /content/curvature-tuning-research/src/curvature-tuning

from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_flowers102",
    train_batch_size=32,
    test_batch_size=64,
    seed=42
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Label min:", labels.min().item())
print("Label max:", labels.max().item())

/content/curvature-tuning-research/src/curvature-tuning


100%|██████████| 345M/345M [00:16<00:00, 20.4MB/s]
100%|██████████| 502/502 [00:00<00:00, 1.44MB/s]
100%|██████████| 15.0k/15.0k [00:00<00:00, 10.3MB/s]

Train samples: 1020
Validation samples: 1020
Test samples: 6149


Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Label min: 3
Label max: 97


# Original Baseline and Single-Parameter Curvature Tuning

The original Curvature Tuning evaluation is first reproduced on Flowers102 using seeds 42, 43, and 44.

Two methods are evaluated:

- **Baseline:** frozen ImageNet-pretrained ResNet-18 with the original ReLU activations.
- **S-CT:** Single-Parameter Curvature Tuning using one global beta value shared across the network.

S-CT performs its original validation-based search for the global beta parameter.

These experiments provide the reference against which the proposed Stage-Wise Curvature Tuning method is compared.

In [ ]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 64

2026-09-24 00:56:26.245 | INFO     | __main__:main:54 - Log file: ./logs/generalization_ct_imagenet_to_flowers102_resnet18_seed42.log
2026-09-24 00:56:26.248 | INFO     | __main__:main:58 - Running on cuda
2026-09-24 00:56:26.823 | INFO     | __main__:main:85 - Testing baseline...
2026-09-24 00:56:26.836 | INFO     | __main__:main:88 - Number of trainable parameters: 52326
2026-09-24 00:56:26.837 | INFO     | __main__:main:89 - Starting transfer learning...
2026-09-24 00:56:39.341 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.757531, Accuracy: 3.12%
2026-09-24 00:56:39.581 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.228337, Val Accuracy: 9.02%
2026-09-24 00:56:39.581 | INFO     | train:linear_probe:190 - New best validation accuracy: 9.02 at epoch 1
2026-09-24 00:56:39.628 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 4.029564, Accuracy: 15.62%
2026-09-24 00:56:39.853 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 3.318797, Val Accuracy: 4

In [ ]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 64

2026-09-24 01:07:43.202 | INFO     | __main__:main:54 - Log file: ./logs/generalization_ct_imagenet_to_flowers102_resnet18_seed43.log
2026-09-24 01:07:43.205 | INFO     | __main__:main:58 - Running on cuda
2026-09-24 01:07:43.783 | INFO     | __main__:main:85 - Testing baseline...
2026-09-24 01:07:43.797 | INFO     | __main__:main:88 - Number of trainable parameters: 52326
2026-09-24 01:07:43.797 | INFO     | __main__:main:89 - Starting transfer learning...
2026-09-24 01:07:55.962 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.895680, Accuracy: 0.00%
2026-09-24 01:07:56.202 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.170027, Val Accuracy: 9.22%
2026-09-24 01:07:56.203 | INFO     | train:linear_probe:190 - New best validation accuracy: 9.22 at epoch 1
2026-09-24 01:07:56.253 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 4.046525, Accuracy: 21.88%
2026-09-24 01:07:56.478 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 3.258339, Val Accuracy: 4

In [ ]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 64

2026-09-24 01:18:57.918 | INFO     | __main__:main:54 - Log file: ./logs/generalization_ct_imagenet_to_flowers102_resnet18_seed44.log
2026-09-24 01:18:57.923 | INFO     | __main__:main:58 - Running on cuda
2026-09-24 01:18:58.705 | INFO     | __main__:main:85 - Testing baseline...
2026-09-24 01:18:58.727 | INFO     | __main__:main:88 - Number of trainable parameters: 52326
2026-09-24 01:18:58.727 | INFO     | __main__:main:89 - Starting transfer learning...
2026-09-24 01:19:10.401 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.680496, Accuracy: 0.00%
2026-09-24 01:19:10.760 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.151015, Val Accuracy: 12.65%
2026-09-24 01:19:10.761 | INFO     | train:linear_probe:190 - New best validation accuracy: 12.65 at epoch 1
2026-09-24 01:19:10.832 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 4.038637, Accuracy: 18.75%
2026-09-24 01:19:11.169 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 3.242137, Val Accuracy:

# Learned 4-Parameter Stage-Wise Curvature Tuning

The Stage-Wise Curvature Tuning method developed during the Beans experiments is now evaluated on Flowers102.

The configuration is transferred without Flowers-specific hyperparameter tuning:

- ImageNet-pretrained ResNet-18
- four trainable stage-wise beta parameters
- beta initialization = **0.78**
- beta learning rate = **0.01**
- curvature coefficient c = **0.5**
- seeds = **42, 43, 44**

The four beta parameters correspond to:

- Stage 1: stem + layer1
- Stage 2: layer2
- Stage 3: layer3
- Stage 4: layer4

The pretrained backbone remains frozen while the downstream classifier and four stage-wise curvature parameters are optimized.

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 42 \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --transfer_train_bs 32 \
    --transfer_test_bs 64

2026-09-24 01:30:21.029 | INFO     | __main__:main:245 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_flowers102_resnet18_seed42_initbeta0.78_betalr0.01.log
2026-09-24 01:30:21.099 | INFO     | __main__:main:261 - Running on cuda
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-24 01:30:21.644 | INFO     | __main__:main:312 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-09-24 01:30:21.645 | INFO     | __main__:main:328 - Trainable curvature parameters: 4
2026-09-24 01:30:21.645 | INFO     | __main__:main:333 - Total trainable parameters: 52330
2026-09-24 01:30:22.931 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.904130, Accuracy: 0.00%
2026-09-24 01:30:37.059 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.024832, Val Accuracy: 13.53%
2026-09-24 01:30:37.060 | INFO     | __main__:transfer:127 - Epoch 1: val_acc=13.53, betas=[0.7831, 0.782, 0.7892, 0.7816]
2026-09-24 01:30:37.075 | INFO     |

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 43 \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --transfer_train_bs 32 \
    --transfer_test_bs 64

2026-09-24 01:35:47.073 | INFO     | __main__:main:245 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_flowers102_resnet18_seed43_initbeta0.78_betalr0.01.log
2026-09-24 01:35:47.142 | INFO     | __main__:main:261 - Running on cuda
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-24 01:35:47.687 | INFO     | __main__:main:312 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-09-24 01:35:47.688 | INFO     | __main__:main:328 - Trainable curvature parameters: 4
2026-09-24 01:35:47.688 | INFO     | __main__:main:333 - Total trainable parameters: 52330
2026-09-24 01:35:48.870 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.813028, Accuracy: 0.00%
2026-09-24 01:36:01.765 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 3.981859, Val Accuracy: 14.80%
2026-09-24 01:36:01.765 | INFO     | __main__:transfer:127 - Epoch 1: val_acc=14.80, betas=[0.7807, 0.7793, 0.7919, 0.7758]
2026-09-24 01:36:01.779 | INFO     

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 44 \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --transfer_train_bs 32 \
    --transfer_test_bs 64

2026-09-24 01:40:59.770 | INFO     | __main__:main:245 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_flowers102_resnet18_seed44_initbeta0.78_betalr0.01.log
2026-09-24 01:40:59.828 | INFO     | __main__:main:261 - Running on cuda
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-24 01:41:00.345 | INFO     | __main__:main:312 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-09-24 01:41:00.346 | INFO     | __main__:main:328 - Trainable curvature parameters: 4
2026-09-24 01:41:00.346 | INFO     | __main__:main:333 - Total trainable parameters: 52330
2026-09-24 01:41:01.563 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.124666, Accuracy: 0.00%
2026-09-24 01:41:14.489 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.002396, Val Accuracy: 14.90%
2026-09-24 01:41:14.490 | INFO     | __main__:transfer:127 - Epoch 1: val_acc=14.90, betas=[0.7879, 0.7769, 0.7769, 0.7702]
2026-09-24 01:41:14.505 | INFO     

## Original Flowers102 Results Summary

The three-seed experiments produced:

| Method | Mean Test Accuracy |
|---|---:|
| Baseline | **80.85 ± 0.27%** |
| S-CT | **81.95 ± 0.31%** |
| Learned SW-CT | **84.75 ± 0.20%** |

The selected global S-CT beta values were:

- Seed 42: **0.82**
- Seed 43: **0.81**
- Seed 44: **0.85**

The learned SW-CT beta vectors were:

- Seed 42: **[0.8612, 0.8189, 0.8402, 0.6970]**
- Seed 43: **[0.8544, 0.8138, 0.8478, 0.6953]**
- Seed 44: **[0.8607, 0.8117, 0.8328, 0.6960]**

Flowers102 shows the clearest improvement from learned Stage-Wise Curvature Tuning among the original three datasets.

Relative to the baseline, learned SW-CT improves mean test accuracy by approximately **+3.90 percentage points**.

Relative to S-CT, learned SW-CT improves mean test accuracy by approximately **+2.80 percentage points**.

# From Learned SW-CT to Formula-SWCT Calibration

The learned SW-CT results demonstrate that stage-specific curvature can substantially improve transfer performance on Flowers102.

However, learned SW-CT still requires optimization of four beta parameters jointly with the downstream classifier and begins from a manually selected initialization.

The later phase of this research investigates whether the stage-wise beta values can instead be predicted directly before downstream training.

The final objective is therefore:

**predict beta first → freeze beta → train the downstream classifier once**

with:

- no target-dataset beta sweep,
- no target-dataset beta optimization,
- no beta = 0.78 initialization requirement.

Because the final Formula-SWCT deployment setting uses fixed beta values, its calibration targets are derived from fixed-beta experiments rather than directly using the jointly learned SW-CT beta vectors above.

## Stage-Isolated Fixed-Beta Calibration

For Formula-SWCT calibration, each ResNet-18 stage is evaluated independently using fixed beta values.

The purpose is to obtain calibration targets that match the final deployment setting, where predicted beta values remain fixed throughout downstream classifier training.

The calibration selection rule is:

> Choose the smallest beta achieving the maximum validation accuracy among the evaluated stage-isolated beta values.

For Flowers102, the resulting stage-wise calibration target is:

**[0.99, 0.95, 0.99, 0.90]**

corresponding to:

- Stage 1: **0.99**
- Stage 2: **0.95**
- Stage 3: **0.99**
- Stage 4: **0.90**

These values are Formula-SWCT calibration targets and are distinct from the beta values learned jointly during the earlier SW-CT experiments.

## Final Formula-SWCT Calibration Set

Flowers102 completes the three-dataset development set used to construct the final Formula-SWCT predictor.

The fixed calibration targets are:

| Dataset | Stage 1 | Stage 2 | Stage 3 | Stage 4 |
|---|---:|---:|---:|---:|
| Beans | 0.85 | 0.85 | 0.85 | 0.80 |
| DTD | 0.95 | 0.95 | 0.99 | 0.99 |
| Flowers102 | 0.99 | 0.95 | 0.99 | 0.90 |

For each dataset, seed, and stage, five descriptors are measured from the frozen ImageNet-pretrained ResNet-18:

1. log activation variance,
2. activation sparsity,
3. log mean absolute activation,
4. log class-separation statistic,
5. normalized stage depth.

The resulting calibration observations are used to fit a ridge-regression predictor with fixed regularization:

**lambda = 0.10**

The fitted Formula-SWCT predictor is then frozen before evaluation on additional target datasets.

# Final Formula-SWCT Protocol

For a new target dataset, Formula-SWCT performs:

Frozen ImageNet-pretrained ResNet-18  
→ collect target-training stage statistics  
→ predict beta_1, beta_2, beta_3, beta_4  
→ clip predictions to [0.70, 0.99]  
→ freeze all four beta values  
→ extract features once  
→ train one downstream linear classifier  
→ select the classifier checkpoint using validation accuracy  
→ evaluate once on the test set

The Formula-SWCT predictor does not use the target test split when determining beta values.

No target-dataset beta sweep or beta optimization is performed.

### Additional Epoch-Length Check

A separate exploratory Flowers102 experiment examined a 30-epoch downstream training schedule.

This experiment is retained separately as an auxiliary sensitivity check and is not used as the canonical benchmark reported in the main three-dataset comparison.

The primary Flowers102 results in this notebook use the standard 20-epoch protocol.